In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.0 MB/s eta 0:00:00


In [ ]:
!pip install torchinfo torchsummary
from torchsummary import summary
from torchinfo import summary

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cp "/content/drive/MyDrive/IDL Project Spring 2025/Data and Code/Organized Data/extracted_data.zip" /content/

In [ ]:
!install unzip

install: missing destination file operand after 'unzip'
Try 'install --help' for more information.


In [ ]:
!unzip /content/extracted_data.zip -d /content/extracted_data/

Archive:  /content/extracted_data.zip
   creating: /content/extracted_data/content/extracted_data/
   creating: /content/extracted_data/content/extracted_data/velocity_data/
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.25_L_4..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.5_L_3..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.75_L_1..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.25_L_3..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_1_L_2..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.25_L_1..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.25_L_2..csv  
  inflating: /content/extracted_data/content/extracted_data/velocity_data/Output_Var_H_0.5_L_2..csv  
  infla

In [ ]:
import numpy as np
import pandas as pd
from torch.optim.lr_scheduler import ReduceLROnPlateau
import csv
import os
import math
from torch_geometric.data import Data, InMemoryDataset
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GCNConv
from tqdm import tqdm
import random

In [ ]:
root = '/content/extracted_data/content/extracted_data'
file_data_Mesh_Node = '/mesh_data/Mesh_Node_From_Mesh_File_H_1_L_4.mph.csv'
file_data_Edge_Element = '/mesh_data/Edge_Elements_Data_H_1_L_4.mph.csv'
file_data_Output_Var = '/velocity_data/Output_Var_H_1_L_4..csv'
file_data_Triangle_Elements = '/mesh_data/Triangle_Elements_Data_H_1_L_4.mph.csv'

H_0.5_L_1
H_0.25_L_1
H_0.25_L_2
H_1_L_1
H_1_L_2

Mean Squared Error (MSE): 0.06507586845746027 (H_1_L_4)
Mean Squared Error (MSE): 0.06555932645964041 (H_1_L_3)
Mean Squared Error (MSE): 0.08993585853803455 (H_0.25_L_3)

In [ ]:
different_parts = file_data_Mesh_Node.split('/')
last_part = different_parts[-1]
different_parts = last_part.split('_')
height = different_parts[-3]
length = different_parts[-1].split('.')
length = length[0]
height = float(height)
length = float(length)
print("height: ",height)
print("length: ", length)

height:  1.0
length:  4.0


## Extract Mesh Node Data from Mesh File and assign Index

In [ ]:
data = pd.read_csv(root+file_data_Mesh_Node)
label_list = list(range(0, data.shape[0]))
data['label'] = label_list
data = data.astype(float)
data.head()

,x,y,label
0,0.000000,0.000000,0.0
1,0.000000,0.012987,1.0
2,0.008463,0.008860,2.0
3,0.012987,0.000000,3.0
4,0.000000,0.025974,4.0


## Triangular elements evaluation

In [ ]:

def read_triangle_edges(csv_file_path):

    edges = []

    file_triangle = pd.read_csv(csv_file_path)
    length = file_triangle.shape[0]
    for i in range(length):

        node_1 = file_triangle.loc[i, 'node_1']
        node_2 = file_triangle.loc[i, 'node_2']
        node_3 = file_triangle.loc[i, 'node_3']

        edges.append((node_1, node_2))
        edges.append((node_2, node_3))
        edges.append((node_3, node_1))

    return edges

def read_edge_elements(csv_file_path):
    edges = []

    file_edge = pd.read_csv(csv_file_path)
    length = file_edge.shape[0]
    for i in range(length):

        node_1 = file_edge.loc[i, 'node_1']
        node_2 = file_edge.loc[i, 'node_2']

        edges.append((node_1, node_2))

    return edges

## Edge Elements Extraction and Combination

In [ ]:
tri_edges_list = read_triangle_edges(root + file_data_Triangle_Elements)
tri_edges_array = np.array(tri_edges_list)  # Shape will be (num_edges, 2)
tri_edges_array = tri_edges_array.T  # Transpose so that each column is an edge

print("Triangular edges array shape:", tri_edges_array.shape)
print("Triangular edges array:")
print(tri_edges_array)

#Collect the list of edge elements from the CSV file
edge_list = read_edge_elements(root + file_data_Edge_Element)
edge_array = np.array(edge_list)
edge_array = edge_array.T

print("Edge array shape:", edge_array.shape)
print("Edge array:")
print(edge_array[:,1:10])

full_adjacency_array = np.concatenate((tri_edges_array, edge_array), axis=1)

edge_raw = []

for i in range(full_adjacency_array.shape[1]):

    edge_raw.append(tuple(full_adjacency_array[:, i]))

edge_unique = set(edge_raw)

total_edges = len(edge_unique)
unique_edges = len(edge_unique)
duplicates_count = total_edges - unique_edges


print("Total number of edges:", total_edges)
print("Number of unique edges:", unique_edges)
print("Duplicates found:", duplicates_count)

Triangular edges array shape: (2, 176058)
Triangular edges array:
[[    2     1     0 ... 29725 29728 29726]
 [    1     0     2 ... 29728 29726 29725]]
Edge array shape: (2, 770)
Edge array:
[[ 3  1  7  4 13  8 20 14 29]
 [ 0  4  3  8  7 14 13 21 20]]
Total number of edges: 176828
Number of unique edges: 176828
Duplicates found: 0


## Output Variable File Post Processing

In [ ]:
def load_mesh(csv_file_path):

    mesh_nodes = []
    file_mesh = pd.read_csv(csv_file_path)
    for i in range(file_mesh.shape[0]):

        x = float(file_mesh.loc[i, 'x'])
        y = float(file_mesh.loc[i, 'y'])
        mesh_nodes.append((x, y))

    return mesh_nodes

def load_velocity(csv_file_path):

    vel_nodes = []
    file_velocity = pd.read_csv(csv_file_path)
    for i in range(file_velocity.shape[0]):
        x = float(file_velocity.loc[i, 'x'])
        y = float(file_velocity.loc[i, 'y'])
        velocity = float(file_velocity.loc[i, 'spf.U_(m/s)'])
        vel_nodes.append((x, y, velocity))

    return vel_nodes

def match_nodes_with_velocity(mesh_nodes, vel_nodes, diff=1e-6):

    matched = []
    for idx, (mx, my) in enumerate(mesh_nodes):
        found_velocity = None
        for (vx, vy, velocity) in vel_nodes:
            if math.isclose(mx, vx, abs_tol=diff) and math.isclose(my, vy, abs_tol=diff):
                found_velocity = velocity
                break
        matched.append((idx, mx, my, found_velocity))
    return matched

In [ ]:
mesh_file = root + file_data_Mesh_Node
vel_file = root + file_data_Output_Var
mesh_nodes = load_mesh(mesh_file)
vel_nodes = load_velocity(vel_file)
matched = match_nodes_with_velocity(mesh_nodes, vel_nodes, diff=1e-6)

header = ["NodeIndex", "x", "y", "velocity"]
output = pd.DataFrame(matched, columns=header)
output.to_csv('combined_data.csv', index=False)

print('save successful in : combined_data.csv')

save successful in : combined_data.csv


## Build Dataset Class

In [ ]:
class MyGraphDataset(InMemoryDataset):
    def __init__(self, combined_file, adjacency_array, transform=None, pre_transform=None):


        super(MyGraphDataset, self).__init__()

        dataset = pd.read_csv(combined_file)
        print(dataset)

        features = dataset[['x', 'y', 'is_edge', 'length', 'height']].values.astype(np.float32)
        test = dataset[['velocity']].values.astype(np.float32)
        self.num_nodes = features.shape[0]

        x = torch.tensor(features, dtype=torch.float)
        y = torch.tensor(test, dtype=torch.float)

        edge_index = torch.tensor(adjacency_array, dtype=torch.long)
        num_edges = edge_array.shape[1]

        num_nodes = self.num_nodes
        indices = torch.randperm(num_nodes)

        train_count = int(0.8 * num_nodes)
        val_count = int(0.1 * num_nodes)
        test_count = num_nodes - train_count - val_count

        train_mask = torch.zeros(num_nodes, dtype=torch.bool)
        val_mask = torch.zeros(num_nodes, dtype=torch.bool)
        test_mask = torch.zeros(num_nodes, dtype=torch.bool)

        train_mask[indices[:train_count]] = True
        val_mask[indices[train_count:train_count+val_count]] = True
        test_mask[indices[train_count+val_count:]] = True


        graph_data = Data(
            x=x,
            y=y,
            edge_index=edge_index,
            train_mask=train_mask,
            val_mask=val_mask,
            test_mask=test_mask
        )

        self.data, self.slices = self.collate([graph_data])

    def __len__(self):
        return self.slices['x'].shape[0]

    def __getitem__(self, idx):
        return self.get(idx)

In [ ]:
dataset = pd.read_csv('combined_data.csv')
is_edge = []
for item in dataset['y']:
    if item in (height, length):
        is_edge.append(1)
    else:
        is_edge.append(0)

In [ ]:
dataset['is_edge'] = is_edge

In [ ]:
dataset['height'] = [height] * dataset.shape[0]
dataset['length'] = [length] * dataset.shape[0]

In [ ]:
dataset.to_csv('combined_data.csv', index=False)

In [ ]:
combined_file = 'combined_data.csv'
adjacency_array = full_adjacency_array
dataset = MyGraphDataset(combined_file, adjacency_array)
data = dataset[0]

       NodeIndex         x         y  velocity  is_edge  height  length
0              0  0.000000  0.000000  4.500000        0     1.0     4.0
1              1  0.000000  0.012987  9.000000        0     1.0     4.0
2              2  0.008463  0.008860  7.857746        0     1.0     4.0
3              3  0.012987  0.000000  0.000000        0     1.0     4.0
4              4  0.000000  0.025974  9.000000        0     1.0     4.0
...          ...       ...       ...       ...      ...     ...     ...
29724      29724  4.000000  0.974026  9.256549        0     1.0     4.0
29725      29725  3.991537  0.991136  6.516911        0     1.0     4.0
29726      29726  3.987013  1.000000  0.000000        1     1.0     4.0
29727      29727  4.000000  0.987013  8.070353        0     1.0     4.0
29728      29728  4.000000  1.000000  0.000000        1     1.0     4.0

[29729 rows x 7 columns]


In [ ]:
'''
train_data = data.clone()
train_data.mask = data.train_mask

val_data = data.clone()
val_data.mask = data.val_mask

test_data = data.clone()
test_data.mask = data.test_mask


print("Number of training nodes:", data.train_mask.sum().item())
print("Number of validation nodes:", data.val_mask.sum().item())
print("Number of test nodes:", data.test_mask.sum().item())
print("Number of training nodes:", train_data.mask.sum().item())
print("Number of validation nodes:", val_data.mask.sum().item())
print("Number of test nodes (after filtering):", test_data.mask.sum().item())
'''

'\ntrain_data = data.clone()\ntrain_data.mask = data.train_mask\n\nval_data = data.clone()\nval_data.mask = data.val_mask\n\ntest_data = data.clone()\ntest_data.mask = data.test_mask\n\n\nprint("Number of training nodes:", data.train_mask.sum().item())\nprint("Number of validation nodes:", data.val_mask.sum().item())\nprint("Number of test nodes:", data.test_mask.sum().item())\nprint("Number of training nodes:", train_data.mask.sum().item())\nprint("Number of validation nodes:", val_data.mask.sum().item())\nprint("Number of test nodes (after filtering):", test_data.mask.sum().item())\n'

In [ ]:
train_data = data.clone()
train_data.mask = data.train_mask.clone()

val_data   = data.clone()
val_data.mask   = data.val_mask.clone()

test_data  = data.clone()
test_data.mask  = data.test_mask.clone()

'''
y = train_data.y.view(-1)
keep = ~((y == 0.0) | (y == 0.25))
train_data.mask &= keep

y = val_data.y.view(-1)
keep = ~((y == 0.0) | (y == 0.25))
val_data.mask &= keep

y = test_data.y.view(-1)
keep = ~((y == 0.0) | (y == 0.25))
test_data.mask &= keep
'''

# 4) Report counts
print("Number of training nodes:", train_data.mask.sum().item())
print("Number of validation nodes:", val_data.mask.sum().item())
print("Number of test nodes (after filtering):", test_data.mask.sum().item())

Number of training nodes: 23783
Number of validation nodes: 2972
Number of test nodes (after filtering): 2974


## Parameters Configuration

In [ ]:
config = {
    'Name': '', # Write your name here
    'subset': 1, # Subset of dataset to use (1.0 == 100% of data)
    'activations': 'Tanh',
    'learning_rate': 0.1,
    'dropout': 0.25,
    'optimizers': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'epochs': 100,
    'batch_size': 1,
    'weight_decay': 0.05,
    'weight_initialization': None, # e.g kaiming_normal, kaiming_uniform, uniform, xavier_normal or xavier_uniform
 }

## Dataloaders

In [ ]:
# Now you can create DataLoaders using these split datasets:
train_loader = DataLoader(
    dataset=[train_data],
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    dataset=[val_data],
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    dataset=[test_data],
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Train dataset samples = {}, batches = {}".format(1, len(train_loader)))
print("Validation dataset samples = {}, batches = {}".format(1, len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(1, len(test_loader)))

Train dataset samples = 1, batches = 1
Validation dataset samples = 1, batches = 1
Test dataset samples = 1, batches = 1


## Visualize graph

## Build GCN

In [ ]:

class Block(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Block, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        self.input_conv = GCNConv(self.input_dim, self.hidden_dim)

        self.block_list =  torch.nn.ModuleList([
                      GCNConv(hidden_dim, hidden_dim),
                      GCNConv(hidden_dim, hidden_dim),
                      GCNConv(hidden_dim, hidden_dim),
                      GCNConv(hidden_dim, hidden_dim),
                      GCNConv(hidden_dim, hidden_dim),

        ])

        self.bn = torch.nn.LayerNorm(hidden_dim)
        self.bn_out = torch.nn.LayerNorm(output_dim)
        self.output_conv = GCNConv(self.hidden_dim, self.output_dim)

    def forward(self, feature_matrix, edge_index):

        x = self.input_conv(feature_matrix, edge_index)
        x = self.bn(x)
        x = F.tanh(x)
        for layer in self.block_list:
            x = layer(x, edge_index)
            x = self.bn(x)
            x = F.tanh(x)

        x = self.output_conv(x, edge_index)
        x = self.bn_out(x)
        x = F.tanh(x)

        return x

class VelocityGCN(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = torch.nn.ModuleList([
            Block(input_dim=5, hidden_dim=128, output_dim=128),
            Block(input_dim=128, hidden_dim=256, output_dim=512),
            Block(input_dim=512, hidden_dim=1024, output_dim=2048),
            Block(input_dim=2048, hidden_dim=2048, output_dim=2048),
            Block(input_dim=2048, hidden_dim=2048, output_dim=2048),
            Block(input_dim=2048, hidden_dim=1024, output_dim=512),
            Block(input_dim=512, hidden_dim=256, output_dim=128),
            Block(input_dim=128, hidden_dim=64, output_dim=32),
        ])

        self.linear = torch.nn.Linear(32, 1)

    def forward(self, feature_matrix, edge_index):

        for block in self.backbone:
            feature_matrix = block(feature_matrix, edge_index)
        out = self.linear(feature_matrix)
        return out


In [ ]:
'''
class Block(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2):
        super(Block, self).__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim_1)
        self.bn1 = torch.nn.LayerNorm(hidden_dim_1)
        self.conv2 = GCNConv(hidden_dim_1, hidden_dim_2)
        self.bn2 = torch.nn.LayerNorm(hidden_dim_2)

    def forward(self, feature_matrix, edge_index):
        x = self.conv1(feature_matrix, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        return x

class VelocityGCN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = torch.nn.ModuleList([
            Block(input_dim=3, hidden_dim_1=16, hidden_dim_2=32),
            Block(input_dim=32, hidden_dim_1=64, hidden_dim_2=128),
            Block(input_dim=128, hidden_dim_1=256, hidden_dim_2=512),
            Block(input_dim=512, hidden_dim_1=1024, hidden_dim_2=1024),
            Block(input_dim=1024, hidden_dim_1=2048, hidden_dim_2=2048),
            Block(input_dim=2048, hidden_dim_1=4096, hidden_dim_2=4096),
            Block(input_dim=4096, hidden_dim_1=2048, hidden_dim_2=1024),
            Block(input_dim=1024, hidden_dim_1=512, hidden_dim_2=256),
            Block(input_dim=256, hidden_dim_1=128, hidden_dim_2=64),
            Block(input_dim=64, hidden_dim_1=32, hidden_dim_2=16),
        ])

        self.linear = torch.nn.Linear(16, 1)

    def forward(self, feature_matrix, edge_index):
        for block in self.backbone:
            feature_matrix = block(feature_matrix, edge_index)
        out = self.linear(feature_matrix)
        return out
'''

'\nclass Block(torch.nn.Module):\n    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2):\n        super(Block, self).__init__()\n\n        self.conv1 = GCNConv(input_dim, hidden_dim_1)\n        self.bn1 = torch.nn.LayerNorm(hidden_dim_1)\n        self.conv2 = GCNConv(hidden_dim_1, hidden_dim_2)\n        self.bn2 = torch.nn.LayerNorm(hidden_dim_2)\n\n    def forward(self, feature_matrix, edge_index):\n        x = self.conv1(feature_matrix, edge_index)\n        x = self.bn1(x)\n        x = F.relu(x)\n        x = self.conv2(x, edge_index)\n        x = self.bn2(x)\n        x = F.relu(x)\n        return x\n\nclass VelocityGCN(torch.nn.Module):\n\n    def __init__(self):\n        super().__init__()\n\n        self.backbone = torch.nn.ModuleList([\n            Block(input_dim=3, hidden_dim_1=16, hidden_dim_2=32),\n            Block(input_dim=32, hidden_dim_1=64, hidden_dim_2=128),\n            Block(input_dim=128, hidden_dim_1=256, hidden_dim_2=512),\n            Block(input_dim=512, 

##  Define Model

In [ ]:
from torchinfo import summary
from torch_geometric.data import Data
import torch

class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super(ModelWrapper, self).__init__()
        self.model = model

    def forward(self, x):
        feature_matrix, edge_index = x  # Unpack the tuple
        return self.model(feature_matrix, edge_index)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VelocityGCN().to(device)

if os.path.exists('graph_model.pth'):
    state = torch.load('graph_model.pth', map_location=device)
    model.load_state_dict(state)
    print(f"Loaded weights from {'graph_model.pth'}")
else:
    print('No pretrained model, train the model from beginning')

data = dataset[0].to(device)

feature_matrix = data.x
edge_index = data.edge_index

wrapped_model = ModelWrapper(model).to(device)

# Print the model summary using torchinfo
summary(wrapped_model, input_data=[(feature_matrix, edge_index)])

No pretrained model, train the model from beginning


Layer (type:depth-idx)                                  Output Shape              Param #
ModelWrapper                                            [29729, 1]                --
├─VelocityGCN: 1-1                                      [29729, 1]                --
│    └─ModuleList: 2-1                                  --                        --
│    │    └─Block: 3-1                                  [29729, 128]              100,352
│    │    └─Block: 3-2                                  [29729, 512]              495,104
│    │    └─Block: 3-3                                  [29729, 2048]             7,878,656
│    │    └─Block: 3-4                                  [29729, 2048]             29,382,656
│    │    └─Block: 3-5                                  [29729, 2048]             29,382,656
│    │    └─Block: 3-6                                  [29729, 512]              7,874,048
│    │    └─Block: 3-7                                  [29729, 128]              493,952
│    │    └─Blo

## Define loss function, Optimizer, and Scheduler

In [ ]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1, weight_decay=5e-4)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=3, threshold=0.0001)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = config['epochs'])

In [ ]:
print(len(train_loader.dataset))
print(data.num_nodes)

1
29729


## Training Function

In [ ]:
def train(data_loader):
    model.train()
    train_losses = 0
    total_nodes = 0
    #batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

    i = 0
    for data in data_loader:
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data.x, data.edge_index)
        # Assume that data.y holds the target velocities for each node.
        loss = criterion(output[train_data.mask], data.y[train_data.mask])
        loss.backward()
        optimizer.step()
        train_losses = train_losses + loss.item()* data.num_nodes
        total_nodes += data.num_nodes
        i = i + 1

    return train_losses / total_nodes

## Validation Function

In [ ]:
def evaluate(model, dataloader, criterion):
    model.eval()  # Set model to evaluation mode.
    total_loss = 0
    total_nodes = 0
    # Disable gradient computation for efficiency.
    i = 0
    with torch.no_grad():
        for data in dataloader:
            data = data.to(device)
            output = model(data.x, data.edge_index)
            loss = criterion(output[val_data.mask], data.y[val_data.mask])
            total_loss = total_loss + loss.item() * data.num_nodes
            total_nodes += data.num_nodes
            i = i + 1

    return total_loss / total_nodes



# Weights and Biases Setup

In [ ]:
import wandb

In [ ]:
wandb.login(key="b80773fd853f81fa052b619bc9e0b0914bc5b8d9") #API Key is in your wandb account, under settings (wandb.ai/settings)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zhuoqil3 (zhuoqili) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Create your wandb run
run = wandb.init(
    name    = "Test-time",
    reinit  = True,
    project = "11785_project",
    config  = config,
)
wandb.watch(model, log="all")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [ ]:
### Save your model architecture as a string with str(model)
model_arch  = str(model)

### Save it in a txt file
arch_file   = open("model_arch.txt", "w")
file_write  = arch_file.write(model_arch)
arch_file.close()

### log it in your wandb run with wandb.save()
#wandb.save('model_arch.txt')

# Experiment

Now, it is time to finally run ablations!

In [ ]:
import torch
import gc
import wandb

# Clear GPU cache and run garbage collection before starting training
torch.cuda.empty_cache()
gc.collect()

# Initialize wandb (make sure you have set up your wandb project)
#wandb.init(project="11785_project", config=config)


for epoch in range(config['epochs']):
    print(f"\nEpoch {epoch+1}/{config['epochs']}")

    # Get the current learning rate
    curr_lr = float(optimizer.param_groups[0]['lr'])

    # Train on the training loader
    train_loss = train(train_loader)
    # print(train_loss)

    # Evaluate on the validation loader
    val_loss = evaluate(model, val_loader, criterion)

    # Print epoch metrics
    print(f"\tTrain Loss: {train_loss:.4f}\tVal Loss: {val_loss:.4f}\tLR: {curr_lr:.7f}")

    # Log metrics to wandb
    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': curr_lr
    })

    # Step the scheduler (here we assume a scheduler that does not require a metric)
    scheduler.step(val_loss)



print("Training complete.")


Epoch 1/100
	Train Loss: 85.6418	Val Loss: 77.1857	LR: 0.1000000

Epoch 2/100
	Train Loss: 77.3112	Val Loss: 64.6098	LR: 0.1000000

Epoch 3/100
	Train Loss: 64.6991	Val Loss: 25.5145	LR: 0.1000000

Epoch 4/100
	Train Loss: 25.4960	Val Loss: 12.6957	LR: 0.1000000

Epoch 5/100
	Train Loss: 12.6219	Val Loss: 4.4277	LR: 0.1000000

Epoch 6/100
	Train Loss: 4.2943	Val Loss: 1.8978	LR: 0.1000000

Epoch 7/100
	Train Loss: 1.7051	Val Loss: 4.3433	LR: 0.1000000

Epoch 8/100
	Train Loss: 4.0990	Val Loss: 8.4928	LR: 0.1000000

Epoch 9/100
	Train Loss: 8.2133	Val Loss: 10.7534	LR: 0.1000000

Epoch 10/100
	Train Loss: 10.4595	Val Loss: 9.8733	LR: 0.1000000

Epoch 11/100
	Train Loss: 9.5848	Val Loss: 7.1899	LR: 0.0900000

Epoch 12/100
	Train Loss: 6.9197	Val Loss: 4.2421	LR: 0.0900000

Epoch 13/100
	Train Loss: 3.9989	Val Loss: 2.3253	LR: 0.0900000

Epoch 14/100
	Train Loss: 2.1128	Val Loss: 1.9254	LR: 0.0900000

Epoch 15/100
	Train Loss: 1.7428	Val Loss: 2.5863	LR: 0.0810000

Epoch 16/100
	Train Lo

In [ ]:
torch.save(model.state_dict(), "graph_model.pth")

# Testing

In [ ]:
def test(model, test_loader):
    model.eval()
    predictions = []
    ground_truth = []

    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            output = model(data.x, data.edge_index)

            test_output = output[test_data.mask]
            test_labels = data.y[test_data.mask]

            predictions.append(test_output.cpu())
            ground_truth.append(test_labels.cpu())

    # Concatenate all predictions and ground truth
    predictions = torch.cat(predictions, dim=0)
    ground_truth = torch.cat(ground_truth, dim=0)

    return predictions, ground_truth

Get predictions and stpre

In [ ]:
predictions, ground_truth= test(model, test_loader)

In [ ]:
print(predictions)

tensor([[8.8497],
        [8.8500],
        [8.8497],
        ...,
        [8.8497],
        [8.8500],
        [8.8501]])


In [ ]:
print(ground_truth)

tensor([[9.0688],
        [9.5745],
        [8.4840],
        ...,
        [9.3847],
        [0.0000],
        [0.0000]])


In [ ]:
# If your tensors are on a GPU, move them to CPU first.
if predictions.is_cuda:
    predictions = predictions.cpu()
if ground_truth.is_cuda:
    ground_truth = ground_truth.cpu()

with open("result.csv", "w+") as f:
    f.write("id, prediction, truth_value\n")
    for i in range(len(predictions)):
        # Convert tensor elements to Python scalars using .item()
        pred = predictions[i].item()
        gt = ground_truth[i].item()
        f.write("{},{},{}\n".format(i, pred, gt))

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(ground_truth, predictions)
print(f"Mean Squared Error (MSE): {mse}")

Mean Squared Error (MSE): 1.7401499182442808


In [ ]:
### Finish your wandb run
run.finish()

epoch,▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
lr,█████▇▇▇▇▇▆▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
train_loss,█▇▆▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
lr,0.00985
train_loss,1.6995
val_loss,1.88938
